# EMOS Tutorial: Channel Material Screening for MOSFET Design

This notebook turns the screening strategy into a minimal, executable workflow that starts with database extraction and ends with device-level ranking using the MOSFET Evaluation feature.

The example is intentionally pragmatic:
- Stage 1 uses a broad, lenient database query to maximize recall.
- Stage 2 uses GBFS to add consistent electronic descriptors.
- Stage 3 uses MatterSim as a relaxation-based stability proxy.
- Stage 4 uses SynthNN to remove candidates with low synthesis likelihood.
- Stage 5 ranks the survivors with the MOSFET solver using a fixed device stack.

The code is kept short on purpose. Increase the batch sizes later if you want a larger search.

In [1]:
import os
import sys
import json
import hashlib
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from pymatgen.core import Structure

workspace_root = Path.cwd().resolve()
while workspace_root != workspace_root.parent:
    if (workspace_root / 'Features').is_dir() and (workspace_root / 'Information_Units').is_dir():
        break
    workspace_root = workspace_root.parent

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from Features.Materials_Exploration.DatabaseExtractor.DatabaseExtractorFeature import DatabaseExtractorFeature
from Features.Electronics_Application.MosfetEvaluator.MosfetEvaluatorFeature import MosfetEvaluatorFeature
from Information_Units.Predictors.Gbfs.GbfsClient import GbfsClient
from Information_Units.Predictors.Mattersim.MattersimPredictor import MattersimPredictor
from Information_Units.Predictors.Synthnn.SynthnnPredictor import SynthnnPredictor


class NotebookLogger:
    def __init__(self):
        self.messages = []

    def log(self, message, level='info'):
        self.messages.append((level, message))


logger = NotebookLogger()


def flatten_cifs(extraction_payload):
    rows = []
    for db_key, db_entry in extraction_payload.get('databases', {}).items():
        payload = db_entry.get('payload', {})
        for cif_text in payload.get('cif_strings', []) if isinstance(payload, dict) else []:
            rows.append({'source': db_key, 'cif': cif_text})
    return rows


def reduced_formula(cif_text):
    try:
        return Structure.from_str(cif_text, fmt='cif').composition.reduced_formula
    except Exception:
        return 'unparsed'


def dedupe_candidates(rows):
    seen = set()
    unique = []
    for row in rows:
        digest = hashlib.sha1(row['cif'].encode('utf-8')).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        unique.append({**row, 'formula': reduced_formula(row['cif']), 'cif_sha1': digest})
    return unique


def force_norm_max(force_array):
    if force_array is None:
        return np.nan
    arr = np.asarray(force_array, dtype=float)
    if arr.size == 0:
        return np.nan
    if arr.ndim == 1:
        return float(np.abs(arr).max())
    return float(np.linalg.norm(arr, axis=1).max())


def print_stage_summary(stage_name, criteria_lines, input_count, output_count, input_unique=None, output_unique=None):
    print(f"\n=== {stage_name} Summary ===")
    print('Selector / Threshold Criteria:')
    for line in criteria_lines:
        print(f"- {line}")
    print(f"Input candidates : {input_count}")
    if input_unique is not None:
        print(f"Input unique formulas : {input_unique}")
    print(f"Output candidates: {output_count}")
    if output_unique is not None:
        print(f"Output unique formulas: {output_unique}")


print(f'Workspace root: {workspace_root}')

Workspace root: /home/eidf204/eidf204/adixit/EMOS


## Optional Service Setup

GBFS and MatterSim are Docker-backed in this workflow. The cell below can start either service when needed; leave the flags disabled when the services are already running.

In [2]:
START_GBFS = False
START_MATTERSIM = False

if START_GBFS:
    subprocess.run(['docker', 'compose', 'up', '-d', 'gbfs-pred'], cwd=workspace_root, check=False)
if START_MATTERSIM:
    subprocess.run(['docker', 'compose', 'up', '-d', 'mattersim'], cwd=workspace_root, check=False)

gbfs = GbfsClient('gbfs', logger=logger)
mattersim = MattersimPredictor('mattersim', logger=logger)
# Refresh health cache so notebook reflects current container state.
MattersimPredictor._health_cache = {'healthy': None, 'checked_at': 0.0}
synthnn = SynthnnPredictor('synthnn', logger=logger)
extractor = DatabaseExtractorFeature(logger=logger)
mosfet = MosfetEvaluatorFeature(logger=logger)

print('GBFS reachable:', gbfs.is_healthy())
print('MatterSim reachable:', mattersim.is_healthy())
print('SynthNN ready')
print('MOSFET evaluator ready')

GBFS reachable: False
MatterSim reachable: False
SynthNN ready
MOSFET evaluator ready


## Stage 1: Broad Candidate Generation with Database Extractor

This stage follows the broad-first, lenient strategy from the Database Extractor application examples. We keep the query permissive: semiconducting materials with moderate band gap, low hull distance, and 2-3 elements.

This is intentionally a *pre-screen*, not the final materials list.

In [3]:
stage1_payload = {
    'batchSize': 400,
    'retrievalMode': 'lenient',
    'targetCompositions': '',
    'queryValues': {
        'band_gap': [0.6, 4.0],
        'hull_distance': [0.0, 0.25],
        'nelements': [2, 3],
    },
    'active_databases': [
        {'value': 'jarvisdft', 'name': 'JARVIS-DFT'},
        {'value': 'alexandria', 'name': 'Alexandria'},
    ],
}

stage1_outputs = extractor.process(stage1_payload)
stage1_rows = dedupe_candidates(flatten_cifs(stage1_outputs['extraction']))
stage1_df = pd.DataFrame(stage1_rows)

print('Stage 1 unique CIF candidates:', len(stage1_df))
print('Stage 1 unique formulas:', stage1_df['formula'].nunique())
display(stage1_df[['formula', 'source']].head(10))

print_stage_summary(
    stage_name='Stage 1 (Database Extraction)',
    criteria_lines=[
        "Retrieval mode = lenient",
        "Databases = JARVIS-DFT + Alexandria",
        "band_gap in [0.6, 4.0] eV",
        "hull_distance in [0.0, 0.25] eV/atom",
        "nelements in [2, 3]",
        "batchSize = 120",
    ],
    input_count='N/A (database search space)',
    output_count=len(stage1_df),
    output_unique=stage1_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered whi

Stage 1 unique CIF candidates: 783
Stage 1 unique formulas: 666


,formula,source
0,Sb,jarvisdft
1,VBr2,jarvisdft
2,ZrN,jarvisdft
3,FeBr2,jarvisdft
4,W2N,jarvisdft
5,Ge2Te,jarvisdft
6,CoI2,jarvisdft
7,CoO2,jarvisdft
8,TlS2,jarvisdft
9,CoO2,jarvisdft



=== Stage 1 (Database Extraction) Summary ===
Selector / Threshold Criteria:
- Retrieval mode = lenient
- Databases = JARVIS-DFT + Alexandria
- band_gap in [0.6, 4.0] eV
- hull_distance in [0.0, 0.25] eV/atom
- nelements in [2, 3]
- batchSize = 120
Input candidates : N/A (database search space)
Output candidates: 783
Output unique formulas: 666


## Stage 2: Descriptor Completion with GBFS

GBFS gives a consistent set of electronic descriptors from the CIFs. We use band gap as the main hard filter, keep dielectric as a soft electrostatics screen, and retain mobility predictions for the final device stage.

The `head(24)` limit keeps the tutorial runtime manageable while still demonstrating the workflow.

In [4]:
stage2_input = stage1_df.to_dict('records')
stage2_predictions = gbfs.predict([row['cif'] for row in stage2_input])['results']

stage2_rows = []
for row, pred in zip(stage2_input, stage2_predictions):
    if pred['status'] != 'ok':
        continue
    props = pred['properties']
    stage2_rows.append({
        **row,
        'bandgap_eV': props.get('bandgap'),
        'dielectric': props.get('dielectric'),
        'mob_n_cm2_Vs': props.get('mob_n'),
        'mob_p_cm2_Vs': props.get('mob_p'),
        'e_form_eV_atom': props.get('e_form'),
    })

stage2_df = pd.DataFrame(stage2_rows)
stage2_df = stage2_df[
    stage2_df['bandgap_eV'].between(0.9, 4.0)
    & (stage2_df['dielectric'] >= 3.0)
].copy()

# Keep one representative CIF per reduced formula to avoid over-weighting near-duplicates.
stage2_df = (
    stage2_df.sort_values(['formula', 'mob_n_cm2_Vs', 'bandgap_eV'], ascending=[True, False, True])
    .sort_values(['mob_n_cm2_Vs', 'bandgap_eV'], ascending=[False, True])
    .head(256)
)

print('Stage 2 retained candidates:', len(stage2_df))
print('Stage 2 unique formulas:', stage2_df['formula'].nunique())
display(stage2_df[['formula', 'bandgap_eV', 'dielectric', 'mob_n_cm2_Vs', 'mob_p_cm2_Vs']])

print_stage_summary(
    stage_name='Stage 2 (GBFS Descriptor Completion)',
    criteria_lines=[
        "Input subset = first 36 Stage-1 candidates (runtime control)",
        "GBFS status must be 'ok'",
        "bandgap_eV in [0.9, 4.0]",
        "dielectric >= 3.0",
        "Drop duplicates by reduced formula (keep highest mob_n, then lower bandgap)",
        "Final selector = top 16 by mob_n desc, bandgap asc",
    ],
    input_count=len(stage2_input),
    output_count=len(stage2_df),
    input_unique=pd.DataFrame(stage2_input)['formula'].nunique(),
    output_unique=stage2_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 2 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while parsing CIF: 6 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  structures = parser.parse_structures(primitive=True)
/home/eidf204/eidf204/adixit/EMOS/Information_Units/Predictors/ContainerPredictorClient.py:82: UserWarning: Issues encountered while 

KeyError: 'bandgap_eV'

## Stage 3: MatterSim Relaxation-Based Stability Proxy

The current MatterSim wrapper does not expose a thermodynamic stability label directly, so this tutorial uses a conservative proxy: successful relaxation with low residual force. This is not equivalent to a full stability consensus, but it is a practical screening step with the current EMOS implementation.

In [ ]:
if not mattersim.is_healthy():
    raise RuntimeError('MatterSim is not reachable. Start it with docker compose up -d mattersim and rerun this cell.')

stage3_input = stage2_df.to_dict('records')
stage3_predictions = mattersim.predict(
    [row['cif'] for row in stage3_input],
    compute_energy=True,
    compute_forces=True,
    compute_stress=False,
    relax=True,
    relax_atoms=True,
    relax_cell=True,
)['results']

stage3_rows = []
for row, pred in zip(stage3_input, stage3_predictions):
    props = pred.get('properties', {})
    max_force = force_norm_max(props.get('relaxed_forces') or props.get('forces'))
    stage3_rows.append({
        **row,
        'mattersim_status': pred.get('status'),
        'relaxed_energy_eV': props.get('relaxed_energy', props.get('energy')),
        'max_force_eV_A': max_force,
    })

stage3_df = pd.DataFrame(stage3_rows)
stage3_df = stage3_df[
    (stage3_df['mattersim_status'] == 'ok')
    & (stage3_df['max_force_eV_A'] <= 0.10)
].copy()
stage3_df = stage3_df.sort_values('max_force_eV_A').head(128)

print('Stage 3 retained candidates:', len(stage3_df))
display(stage3_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'max_force_eV_A', 'relaxed_energy_eV']])

print_stage_summary(
    stage_name='Stage 3 (MatterSim Stability Proxy)',
    criteria_lines=[
        "MatterSim container must be healthy",
        "Prediction status == 'ok'",
        "Residual force threshold: max_force_eV_A <= 0.10",
        "Final selector = top 128 with lowest max_force_eV_A",
    ],
    input_count=len(stage3_input),
    output_count=len(stage3_df),
    input_unique=pd.DataFrame(stage3_input)['formula'].nunique(),
    output_unique=stage3_df['formula'].nunique(),
)

Stage 3 retained candidates: 128


,formula,bandgap_eV,mob_n_cm2_Vs,max_force_eV_A,relaxed_energy_eV
55,Pm,1.088547,48.247762,0.000000e+00,-4.736027
53,B,1.509183,49.481261,0.000000e+00,-5.391483
127,F2,2.548695,1.156171,0.000000e+00,-1.436110
128,I,1.213857,1.043360,0.000000e+00,-1.345587
80,SrY,0.971241,25.497913,6.426761e-08,-7.653811
...,...,...,...,...,...
32,MoS2,1.729899,121.552877,8.110315e-03,-21.722237
123,PtO2,1.872357,1.779435,8.113742e-03,-17.355288
37,B2O,3.478503,97.084230,8.550126e-03,-20.260509
59,MgB2,1.317817,47.285961,8.651677e-03,-13.482382



=== Stage 3 (MatterSim Stability Proxy) Summary ===
Selector / Threshold Criteria:
- MatterSim container must be healthy
- Prediction status == 'ok'
- Residual force threshold: max_force_eV_A <= 0.10
- Final selector = top 128 with lowest max_force_eV_A
Input candidates : 129
Input unique formulas : 106
Output candidates: 128
Output unique formulas: 105


## Stage 4: Synthesizability Filtering with SynthNN

A threshold of 0.70 matches the SynthNN default classification rule and gives a shortlist that is more experimentally realistic without being overly restrictive.

In [ ]:
stage4_input = stage3_df.to_dict('records')
stage4_predictions = synthnn.predict([row['cif'] for row in stage4_input])['results']

stage4_rows = []
for row, pred in zip(stage4_input, stage4_predictions):
    props = pred.get('properties', {})
    stage4_rows.append({
        **row,
        'synthesizable': props.get('synthesizable'),
        'synthesizability_score': props.get('synthesizability_score'),
    })

stage4_df = pd.DataFrame(stage4_rows)
stage4_df = stage4_df[stage4_df['synthesizability_score'] >= 0.70].copy()
stage4_df = stage4_df.sort_values('synthesizability_score', ascending=False).head(10).unique(subset='formula', keep='first')

print('Stage 4 retained candidates:', len(stage4_df))
display(stage4_df[['formula', 'bandgap_eV', 'mob_n_cm2_Vs', 'synthesizability_score']])

print_stage_summary(
    stage_name='Stage 4 (SynthNN Synthesizability)',
    criteria_lines=[
        "SynthNN inference on all Stage-3 candidates",
        "synthesizability_score >= 0.70",
        "Final selector = top 32 by synthesizability_score desc",
    ],
    input_count=len(stage4_input),
    output_count=len(stage4_df),
    input_unique=pd.DataFrame(stage4_input)['formula'].nunique(),
    output_unique=stage4_df['formula'].nunique(),
)

/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))
2026-11-10 16:29:34.750533: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-10 16:29:34.756318: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-10 16:29:34.770192: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794328174.792137  538535 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794328174.798822  538535 cuda_blas.cc:1418] Unable t

Instructions for updating:
Use `tf.global_variables_initializer` instead.


2026-11-10 16:29:50.156253: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1794328190.185814  538535 mlir_graph_optimization_pass.cc:401] MLIR V1 optimization pass is not enabled
/home/eidf204/eidf204/adixit/EMOS/emos_env/lib/python3.12/site-packages/pymatgen/core/composition.py:1373: FutureWarning: gcd is deprecated, and will be removed on 2028-01-01
Use math.gcd instead.
  factor = abs(gcd(*(int(i) for i in sym_amt.values())))


Stage 4 retained candidates: 32


,formula,bandgap_eV,mob_n_cm2_Vs,synthesizability_score
37,AgI,1.012800,103.965633,0.9640
114,TaSe2,1.076894,38.588791,0.9272
53,TaSe2,1.076894,38.588791,0.9272
60,TaSe2,1.076894,38.588791,0.9272
71,TaSe2,1.076894,38.588791,0.9272
78,HgIBr,1.689387,9.158498,0.9115
89,YbI2,2.399830,21.348433,0.9072
55,HgI2,1.043888,9.586455,0.8952
18,ZrSbRh,2.024293,34.129063,0.8804
74,TiO2,2.328054,15.741857,0.8627



=== Stage 4 (SynthNN Synthesizability) Summary ===
Selector / Threshold Criteria:
- SynthNN inference on all Stage-3 candidates
- synthesizability_score >= 0.70
- Final selector = top 32 by synthesizability_score desc
Input candidates : 128
Input unique formulas : 105
Output candidates: 32
Output unique formulas: 17


## Stage 5: Device-Level Ranking with MOSFET Evaluation

The MOSFET solver uses the candidate-specific channel descriptors where we have them now: band gap, relative permittivity, and carrier mobilities. Missing transport parameters are filled with fixed baseline values so that ranking is driven by the candidate-dependent properties rather than by inconsistent assumptions.

The ranking rule follows the strategy document: enforce a practical off-current and threshold window first, then prioritize `Ion/Ioff`, and finally use `Ion` as the secondary sort key.

In [ ]:
BASE_MOSFET_INPUTS = {
    'channelLengthNm': 14,
    'sourceDrainLengthNm': 4,
    'oxideThicknessNm': 1.0,
    'channelThicknessNm': 4.0,
    'temperatureK': 300,
    'gateWorkFunctionEv': 3.65,
    'sdWorkFunctionEv': 0.0,
    'channelDopingCm3': -1e15,
    'sourceDrainDopingCm3': 1e20,
    'gateVoltageSweepStartV': 0.0,
    'gateVoltageSweepStopV': 0.7,
    'numberOfGatePoints': 14,
    'drainVoltageSweepStartV': 0.0,
    'drainVoltageSweepStopV': 0.7,
    'numberOfDrainPoints': 13,
    'channelNc': 2.8e25,
    'channelNv': 1.04e25,
    'channelXiEv': 4.05,
    'channelVsatN': 2e5,
    'channelVsatP': 2e5,
    'channelPowN': 2.0,
    'channelPowP': 1.0,
    'insulatorNc': 1.0,
    'insulatorNv': 1.0,
    'insulatorEpsRel': 3.9,
    'insulatorUn': 1e-3,
    'insulatorUp': 1e-3,
    'insulatorXiEv': 0.9,
    'insulatorEgEv': 9.0,
    'insulatorVsatN': 2e5,
    'insulatorVsatP': 2e5,
    'insulatorPowN': 2.0,
    'insulatorPowP': 1.0,
}

def build_mosfet_payload(row):
    # GBFS mobility is in cm^2/Vs; the solver expects m^2/Vs.
    mu_n = max(float(row['mob_n_cm2_Vs']) * 1e-4, 1e-5)
    mu_p = max(float(row['mob_p_cm2_Vs']) * 1e-4, 1e-5)
    return {
        **BASE_MOSFET_INPUTS,
        'channelEgEv': float(np.clip(row['bandgap_eV'], 0.8, 4.0)),
        'channelEpsRel': float(np.clip(row['dielectric'], 2.0, 40.0)),
        'channelUn': mu_n,
        'channelUp': mu_p,
    }

stage5_rows = []
for row in stage4_df.to_dict('records'):
    outputs = mosfet.process(build_mosfet_payload(row))
    metrics = outputs['key_metrics']
    stage5_rows.append({
        'formula': row['formula'],
        'bandgap_eV': row['bandgap_eV'],
        'mob_n_cm2_Vs': row['mob_n_cm2_Vs'],
        'synthesizability_score': row['synthesizability_score'],
        'Id_on_uA_per_um': metrics['Id_on_uA_per_um'],
        'Id_off_uA_per_um': metrics['Id_off_uA_per_um'],
        'Vth_approx_V': metrics['Vth_approx_V'],
        'Ion_Ioff_ratio': metrics['Ion_Ioff_ratio'],
    })

stage5_df = pd.DataFrame(stage5_rows)
stage5_df['passes_device_screen'] = (
    (stage5_df['Id_off_uA_per_um'] <= 1e-2)
    & stage5_df['Vth_approx_V'].between(0.15, 0.65)
)
stage5_df['Vth_distance'] = (stage5_df['Vth_approx_V'] - 0.35).abs()
stage5_ranked = stage5_df.sort_values(
    ['passes_device_screen', 'Ion_Ioff_ratio', 'Id_on_uA_per_um', 'Vth_distance'],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(stage5_ranked)

print_stage_summary(
    stage_name='Stage 5 (MOSFET Device Ranking)',
    criteria_lines=[
        "Use fixed MOSFET geometry/bias stack for all candidates",
        "Map candidate properties into channelEgEv, channelEpsRel, channelUn, channelUp",
        "Device screen: Id_off_uA_per_um <= 1e-2 and 0.15 <= Vth_approx_V <= 0.65",
        "Ranking selector: passes_device_screen desc, Ion_Ioff_ratio desc, Id_on_uA_per_um desc, Vth_distance asc",
    ],
    input_count=len(stage4_df),
    output_count=len(stage5_ranked),
    input_unique=stage4_df['formula'].nunique(),
    output_unique=stage5_ranked['formula'].nunique(),
)

Bad pipe message: %s [b'u; Linux x86_64; rv:158.0) Gecko/20100101 Firefox/158.0\r\nAccept: text/html,application/xhtml+xml,application/']


## Notes

- This notebook is a tutorial-sized example, so each stage uses small candidate caps to keep runtime reasonable.
- Stage 3 is a relaxation-based proxy for stability, not a full thermodynamic stability analysis.
- Stage 5 uses the MOSFET evaluator as it exists today, so the final ranking is based on `Ion/Ioff`, `Ion`, and a practical `Vth` window rather than on SS or DIBL.
- Once EMOS exposes richer stability and device metrics, the same structure can be tightened without changing the notebook layout.